# Live demo: one prefix program, trained in front of you

Trains a 4-slot prefix for *translation into Russian* on the frozen Qwen2.5-1.5B (300 steps on 512 sentence pairs; about 2 minutes on an A100, 8 on a T4), scores it on 256 held-out FLORES+ sentences, and places the result on the stored Design A curve from the full runs. Then it looks at what the four vectors are, nearest-token-wise, which is the waywardness point: the program works, and it does not read as words.

Needs a GPU runtime and a Hugging Face token with FLORES+ access (Colab secret `HF_TOKEN`).

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    if not os.path.exists("/content/prefix-programs"):
        token = userdata.get("GITHUB_TOKEN")
        subprocess.run(f"git clone -q -b prefixlab https://{token}@github.com/sirotenkogeorgii/prefix-programs.git /content/prefix-programs", shell=True, check=True)
        subprocess.run("pip install -q -e /content/prefix-programs", shell=True, check=True)
    os.chdir("/content/prefix-programs")
else:
    os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
import torch; print("cuda:", torch.cuda.is_available(), "| cwd:", os.getcwd())

In [ ]:
MODEL, CELL, K, STEPS = "Qwen/Qwen2.5-1.5B", ("translation", "ru"), 4, 300
cmd = (f"python scripts/run_sweep.py --design A --model {MODEL} --tasks {CELL[0]} --langs {CELL[1]} --ks {K} --seeds 0 --conds none "
       f"--steps {STEPS} --eval-every 50 --max-train 512 --max-dev 128 --max-test 256 --baselines "
       f"--results results_demo/sweep.csv --ckpt-dir checkpoints_demo")
print(cmd); !{cmd}

## Where does the live prefix land on the stored curve?

The stored no-prefix loss and floor for this cell come from the full runs (`results_sync/sweep.csv`); the demo prefix is scored on a 256-sentence subset, so expect it a little above the stored k = 4 point (fewer steps, fewer training pairs).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from prefixlab.analysis import load
T = load.load_tables("results_sync"); fl = load.floors(T["sweep"]); sw = load.attach_D(T["sweep"], fl)
cell = f"{CELL[0]}-{CELL[1]}"
demo = pd.read_csv("results_demo/sweep.csv"); demo = load.attach_D(demo, fl)
row = demo[(demo.design == "A") & (demo.cell == cell)].iloc[-1]
stored = sw[(sw.design == "A") & (sw.cell == cell) & (sw.cond == "none")].groupby("k").D.mean()
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.plot(stored.index, stored.values, marker="o", label="stored Design A (1500 steps, full data)")
ax.scatter([K], [row.D], color="red", zorder=5, s=60, label=f"live: {STEPS} steps, 512 pairs")
ax.axhline(0.10, color="grey", ls="--", lw=0.6); ax.axhline(0.25, color="grey", ls="--", lw=0.6)
ax.set_xscale("log", base=2); ax.set_xlabel("slots k"); ax.set_ylabel("D-hat"); ax.set_title(cell); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()
print(f"live prefix: {row.test_bits_per_token:.3f} bits/token, D-hat = {row.D:.3f}; no prefix {fl.set_index('cell').loc[cell, 'L_none']:.3f}, floor {fl.set_index('cell').loc[cell, 'L_floor']:.3f}")

In [ ]:
import json
hist = json.load(open(row.history)) if isinstance(row.history, str) and os.path.exists(row.history) else []
pd.DataFrame(hist)[["step", "dev_bits_per_token"]].set_index("step").plot(marker="o", figsize=(5, 2.6), title="dev bits/token during training", legend=False); plt.show()

## What is in the four vectors?

Nearest vocabulary tokens by cosine to each slot. Khashabi et al. (2022): a working soft prompt need not read as anything. This is the exploratory, do-not-over-interpret view.

In [ ]:
import torch
from prefixlab.model import load_model
model, tok = load_model(MODEL)
E = model.get_input_embeddings().weight.float()
state = torch.load(row.ckpt, map_location=E.device)
P = state["weight"].float().to(E.device) if "weight" in state else next(iter(state.values())).float().to(E.device)
En = torch.nn.functional.normalize(E, dim=1); Pn = torch.nn.functional.normalize(P, dim=1)
for i in range(P.shape[0]):
    sims = En @ Pn[i]; top = torch.topk(sims, 6)
    print(f"slot {i}: norm {P[i].norm():.2f} | " + ", ".join(f"{tok.decode([j])!r} ({s:.2f})" for s, j in zip(top.values.tolist(), top.indices.tolist())))
print("typical embedding norm:", E.norm(dim=1).median().item())

## And without the prefix?

Greedy continuations for two dev sentences, with and without the live prefix. The prefix is a program: the same frozen weights, a different behaviour.

In [ ]:
from prefixlab.tasks import cell_pairs, Cell
from prefixlab.cli_common import load_flores_for
import argparse
args = argparse.Namespace(flores_dir=None, mlama_dir="data_raw/mlama", hf_token=os.environ.get("HF_TOKEN"), tasks=[CELL[0]], langs=[CELL[1]], dev_frac=0.2, split_seed=0, max_train=None, max_dev=None, max_test=None)
dev_rows, devtest_rows = load_flores_for(args, [CELL[0]], [CELL[1]])
pairs = cell_pairs(Cell(*CELL), dev_rows, devtest_rows, "data_raw/mlama")["test"][:2]
@torch.no_grad()
def generate(text, prefix=None, n=25):
    ids = tok(text + "\n", return_tensors="pt", add_special_tokens=False).input_ids.to(E.device)
    emb = model.get_input_embeddings()(ids)
    if prefix is not None:
        emb = torch.cat([prefix.unsqueeze(0).to(emb.dtype), emb], dim=1)
    out = model.generate(inputs_embeds=emb, max_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0], skip_special_tokens=True)
for src, tgt in pairs:
    print("SOURCE   :", src); print("REFERENCE:", tgt)
    print("no prefix:", generate(src).split("\n")[0]); print("prefix   :", generate(src, P).split("\n")[0]); print()